In [1]:
# =============================================================================
# Cell 1: Setup & Dependencies
# =============================================================================
# Transformer Baseline Notebook for HIA-PAL-SMCNN paper
# Runs SpectralFormer and SSFTT under IDENTICAL conditions as the main paper
# Same datasets, same splits, same seeds, same metrics

!pip install -q einops timm

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import json
import time
import random
import scipy.io as sio
import urllib.request
from sklearn.metrics import accuracy_score, cohen_kappa_score, precision_score, recall_score, f1_score, confusion_matrix
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

RESULTS_DIR = '/kaggle/working/transformer_results/'
os.makedirs(RESULTS_DIR, exist_ok=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incomp

In [2]:
# =============================================================================
# Cell 2: Data Loader — IDENTICAL to main paper
# =============================================================================
# Same datasets, same splits, same seeds, same preprocessing
# This ensures FAIR comparison (reviewer requirement)

import glob

DATASET_INFO = {
    'IndianPines': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/6/67/Indian_pines_corrected.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/c/c4/Indian_pines_gt.mat',
        'data_key': 'indian_pines_corrected',
        'gt_key':   'indian_pines_gt',
        'num_classes': 16
    },
    'PaviaU': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/e/ee/PaviaU.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/5/50/PaviaU_gt.mat',
        'data_key': 'paviaU',
        'gt_key':   'paviaU_gt',
        'num_classes': 9
    },
    'Botswana': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/7/72/Botswana.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/5/58/Botswana_gt.mat',
        'data_key': 'Botswana',
        'gt_key':   'Botswana_gt',
        'num_classes': 14
    },
    'KSC': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/2/26/KSC.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/a/a6/KSC_gt.mat',
        'data_key': 'KSC',
        'gt_key':   'KSC_gt',
        'num_classes': 13
    },
    'WHU_Hi': {
        'data_url': 'KAGGLE_INPUT',
        'gt_url':   'KAGGLE_INPUT',
        'data_key': 'WHU_Hi_HanChuan',
        'gt_key':   'WHU_Hi_HanChuan_gt',
        'num_classes': 16
    },
    'Salinas': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/a/a3/Salinas_corrected.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat',
        'data_key': 'salinas_corrected',
        'gt_key':   'salinas_gt',
        'num_classes': 16
    },
    'Houston2013': {
        'data_url': 'KAGGLE_INPUT',
        'gt_url':   'KAGGLE_INPUT',
        'data_key': 'Houston',
        'gt_key':   'Houston_gt',
        'num_classes': 15
    }
}

def download_dataset(name):
    info = DATASET_INFO[name]
    os.makedirs('datasets', exist_ok=True)
    data_path = f"datasets/{name}.mat"
    gt_path   = f"datasets/{name}_gt.mat"

    search_patterns = {
        'IndianPines': (['*ndian*pines*corrected*.mat', '*Indian_pines.mat'], ['*ndian*pines*gt*.mat']),
        'PaviaU':      (['*aviaU.mat'], ['*aviaU*gt*.mat']),
        'Salinas':     (['*alinas_corrected*.mat', '*alinas.mat'], ['*alinas_gt*.mat']),
        'Botswana':    (['*otswana.mat'], ['*otswana_gt*.mat']),
        'KSC':         (['*KSC.mat'], ['*KSC_gt*.mat']),
        'Houston2013': (['*ouston*.mat', '*ouston*data*.mat', '*Houston13.mat'], ['*ouston*gt*.mat', '*ouston*label*.mat', '*Houston13_7gt.mat']),
        'WHU_Hi':      (['*WHU*Han*[!g][!t].mat', '*HanChuan.mat'], ['*HanChuan*gt*.mat']),
    }

    found_data = found_gt = None
    data_pats, gt_pats = search_patterns.get(name, ([], []))
    
    import glob
    def _find_in_kaggle_input(pattern):
        matches = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
        return matches[0] if matches else None

    for pat in data_pats:
        found_data = _find_in_kaggle_input(pat)
        if found_data: break
    for pat in gt_pats:
        found_gt = _find_in_kaggle_input(pat)
        if found_gt: break

    if found_data and found_gt:
        print(f"  Found {name} in Kaggle Input:")
        print(f"    Data: {found_data}")
        print(f"    GT:   {found_gt}")
        return found_data, found_gt

    if info['data_url'] == 'KAGGLE_INPUT':
        raise FileNotFoundError(f"{name} not found in /kaggle/input/")
    
    if not os.path.exists(data_path):
        print(f"  Downloading {name} data...")
        import urllib.request
        try:
            urllib.request.urlretrieve(info['data_url'], data_path)
        except Exception as e:
            print(f"  Failed to download data: {e}")
            raise
    if not os.path.exists(gt_path):
        print(f"  Downloading {name} GT...")
        import urllib.request
        try:
            urllib.request.urlretrieve(info['gt_url'], gt_path)
        except Exception as e:
            print(f"  Failed to download GT: {e}")
            raise
            
    return data_path, gt_path

def load_dataset(name):
    dp, gp = download_dataset(name)
    info = DATASET_INFO[name]
    
    def _get(mat_file, key):
        data = sio.loadmat(mat_file)
        if key in data: return data[key]
        arrays = {k: v for k, v in data.items() if not k.startswith('__') and hasattr(v, 'shape')}
        k = max(arrays, key=lambda k: arrays[k].size)
        return arrays[k]
    
    X = _get(dp, info['data_key'])
    y = _get(gp, info['gt_key'])
    print(f"  Loaded {name}: X={X.shape}, y={y.shape}, classes={info['num_classes']}")
    return X, y

def normalize(X):
    """Per-band normalization to [0, 1]."""
    X = X.astype(np.float32)
    for b in range(X.shape[2]):
        band = X[:, :, b]
        mn, mx = band.min(), band.max()
        if mx > mn:
            X[:, :, b] = (band - mn) / (mx - mn)
    return X

def pad_with_zeros(X, margin):
    h, w, b = X.shape
    padded = np.zeros((h + 2*margin, w + 2*margin, b), dtype=X.dtype)
    padded[margin:margin+h, margin:margin+w, :] = X
    return padded

def create_disjoint_patches(X, y, window_size=15, train_ratio=0.05, seed=42):
    """
    IDENTICAL split to main paper — pixel-level disjoint, no data leakage.
    """
    rng = np.random.RandomState(seed)
    margin = (window_size - 1) // 2
    padded_X = pad_with_zeros(X, margin)
    
    X_train, y_train = [], []
    X_test, y_test = [], []
    
    classes = np.unique(y[y > 0])
    for c in classes:
        idx = np.argwhere(y == c)
        rng.shuffle(idx)
        n_train = max(1, int(len(idx) * train_ratio))
        for r, ci in idx[:n_train]:
            X_train.append(padded_X[r:r+window_size, ci:ci+window_size, :])
            y_train.append(int(c - 1))
        for r, ci in idx[n_train:]:
            X_test.append(padded_X[r:r+window_size, ci:ci+window_size, :])
            y_test.append(int(c - 1))
    
    X_train = np.array(X_train, dtype=np.float32)
    y_train = np.array(y_train, dtype=np.int64)
    X_test = np.array(X_test, dtype=np.float32)
    y_test = np.array(y_test, dtype=np.int64)
    print(f"  Split (seed={seed}): {len(y_train)} train, {len(y_test)} test")
    return X_train, X_test, y_train, y_test

print("Data loader ready (identical protocol to main paper).")


Data loader ready (identical protocol to main paper).


In [3]:
# =============================================================================
# Cell 3: SpectralFormer Model
# =============================================================================
# Reference: Hong et al., "SpectralFormer: Rethinking Hyperspectral Image
#            Classification with Transformers," IEEE TGRS, 2022.
#
# Adapted for FAIR comparison: same patch size (15×15), same input format

import math
from einops import rearrange

class SpectralEmbedding(nn.Module):
    """Embeds spectral bands as tokens with positional encoding."""
    def __init__(self, num_bands, patch_size, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim
        # Each band's spatial patch → flattened → projected to embed_dim
        self.proj = nn.Linear(patch_size * patch_size, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, num_bands + 1, embed_dim) * 0.02)
        self.norm = nn.LayerNorm(embed_dim)
    
    def forward(self, x):
        # x: (B, H, W, Bands) → (B, Bands, H, W)
        x = x.permute(0, 3, 1, 2)
        B, C, H, W = x.shape
        # Each band becomes a token: (B, C, H*W) → (B, C, embed_dim)
        x = x.reshape(B, C, H * W)
        x = self.proj(x)  # (B, C, embed_dim)
        
        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)  # (B, C+1, embed_dim)
        
        # Add positional encoding (truncate or pad if needed)
        if x.size(1) <= self.pos_embed.size(1):
            x = x + self.pos_embed[:, :x.size(1), :]
        else:
            # Interpolate positional embeddings for larger band counts
            pos = F.interpolate(
                self.pos_embed.permute(0, 2, 1), size=x.size(1), mode='linear'
            ).permute(0, 2, 1)
            x = x + pos
        
        return self.norm(x)


class CrossLayerAdaptiveFusion(nn.Module):
    """Cross-layer adaptive fusion (CAF) from SpectralFormer."""
    def __init__(self, embed_dim):
        super().__init__()
        self.gate = nn.Sequential(
            nn.Linear(embed_dim * 2, embed_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x_current, x_previous):
        combined = torch.cat([x_current, x_previous], dim=-1)
        gate = self.gate(combined)
        return gate * x_current + (1 - gate) * x_previous


class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.mlp(self.norm2(x))
        return x


class SpectralFormer(nn.Module):
    """
    SpectralFormer with Cross-Layer Adaptive Fusion.
    
    Architecture:
        Spectral Embedding → [TransformerBlock + CAF] × depth → CLS → FC
    """
    def __init__(self, num_classes, num_bands, patch_size=15,
                 embed_dim=64, depth=4, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.embed = SpectralEmbedding(num_bands, patch_size, embed_dim)
        
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])
        self.cafs = nn.ModuleList([
            CrossLayerAdaptiveFusion(embed_dim)
            for _ in range(depth - 1)  # CAF between consecutive layers
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        self.dropout = nn.Dropout(dropout)
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    
    def forward(self, x):
        # x: (B, H, W, Bands)
        x = self.embed(x)  # (B, bands+1, embed_dim)
        
        prev = x
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i > 0:  # Apply CAF from second layer onwards
                x = self.cafs[i - 1](x, prev)
            prev = x
        
        x = self.norm(x)
        cls_token = x[:, 0]  # CLS token
        return self.head(self.dropout(cls_token))
    
    def extract_features(self, x):
        x = self.embed(x)
        prev = x
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i > 0:
                x = self.cafs[i - 1](x, prev)
            prev = x
        x = self.norm(x)
        return x[:, 0]


def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# Quick test
_test = SpectralFormer(num_classes=16, num_bands=200, patch_size=15)
print(f"SpectralFormer ready. Test params: {count_params(_test)/1e6:.3f}M")
del _test


SpectralFormer ready. Test params: 0.253M


In [4]:
# =============================================================================
# Cell 4: SSFTT Model
# =============================================================================
# Reference: Sun et al., "Spectral-Spatial Feature Tokenization Transformer
#            for HSI Classification," IEEE TGRS, 2022.
#
# SSFTT = Spectral-Spatial Feature Tokenization + Gaussian Weighted Transformer
# Hybrid: 3D-Conv tokenizer → Transformer encoder → classifier

class SpectralSpatialTokenizer(nn.Module):
    """
    3D Conv tokenizer: extracts spectral-spatial tokens from HSI patches.
    Conv3D reduces spectral dim while preserving spatial structure,
    then tokens are created by flattening the spatial grid.
    """
    def __init__(self, num_bands, embed_dim=64):
        super().__init__()
        # Spectral reduction via 3D convolution
        self.conv3d_1 = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=(7, 3, 3), stride=(2, 1, 1), padding=(3, 1, 1)),
            nn.BatchNorm3d(8),
            nn.ReLU(inplace=True)
        )
        self.conv3d_2 = nn.Sequential(
            nn.Conv3d(8, 16, kernel_size=(5, 3, 3), stride=(2, 1, 1), padding=(2, 1, 1)),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True)
        )
        self.conv3d_3 = nn.Sequential(
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(2, 1, 1), padding=(1, 1, 1)),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True)
        )
        
        # After 3 stride-2 reductions: spectral_dim ≈ num_bands / 8
        # We'll flatten spectral × channels and project to embed_dim
        self._probe_dim = None  # Will be computed in forward
        self.proj = None  # Lazy init
        self.embed_dim = embed_dim
    
    def _init_proj(self, feat_dim):
        self.proj = nn.Linear(feat_dim, self.embed_dim).to(
            next(self.conv3d_1.parameters()).device
        )
        nn.init.trunc_normal_(self.proj.weight, std=0.02)
        nn.init.zeros_(self.proj.bias)
    
    def forward(self, x):
        # x: (B, H, W, Bands) → (B, 1, Bands, H, W)
        x = x.permute(0, 3, 1, 2).unsqueeze(1)
        
        x = self.conv3d_1(x)   # (B, 8, Bands/2, H, W)
        x = self.conv3d_2(x)   # (B, 16, Bands/4, H, W)
        x = self.conv3d_3(x)   # (B, 32, Bands/8, H, W)
        
        B, C, D, H, W = x.shape
        # Reshape: spatial tokens = H*W, each token = C*D features
        x = x.permute(0, 3, 4, 1, 2)  # (B, H, W, C, D)
        x = x.reshape(B, H * W, C * D)  # (B, H*W, C*D)
        
        # Lazy projection init
        if self.proj is None:
            self._init_proj(C * D)
        
        x = self.proj(x)  # (B, H*W, embed_dim)
        return x, H, W


class GaussianWeightedSelfAttention(nn.Module):
    """
    Self-attention with Gaussian position bias (SSFTT's key contribution).
    Spatial tokens that are physically closer get higher attention.
    """
    def __init__(self, embed_dim, num_heads, spatial_size=15):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.proj = nn.Linear(embed_dim, embed_dim)
        
        # Gaussian positional bias
        self.sigma = nn.Parameter(torch.tensor(2.0))
        coords = torch.stack(torch.meshgrid(
            torch.arange(spatial_size), torch.arange(spatial_size), indexing='ij'
        ), dim=-1).float().reshape(-1, 2)
        dist = torch.cdist(coords, coords, p=2)
        self.register_buffer('dist_matrix', dist)
    
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        
        # Add Gaussian spatial bias
        if N <= self.dist_matrix.size(0):
            gauss_bias = torch.exp(-self.dist_matrix[:N, :N] ** 2 / (2 * self.sigma ** 2 + 1e-6))
            attn = attn + gauss_bias.unsqueeze(0).unsqueeze(0)
        
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)


class SSFTTBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.1, spatial_size=15):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = GaussianWeightedSelfAttention(embed_dim, num_heads, spatial_size)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim),
            nn.Dropout(dropout)
        )
    
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class SSFTT(nn.Module):
    """
    Spectral-Spatial Feature Tokenization Transformer.
    
    Architecture:
        3D Conv Tokenizer → [Gaussian-Weighted Transformer] × depth → GAP → FC
    """
    def __init__(self, num_classes, num_bands, patch_size=15,
                 embed_dim=64, depth=2, num_heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.tokenizer = SpectralSpatialTokenizer(num_bands, embed_dim)
        
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        
        self.blocks = nn.ModuleList([
            SSFTTBlock(embed_dim, num_heads, mlp_ratio, dropout, patch_size)
            for _ in range(depth)
        ])
        
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        self.dropout_layer = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (B, H, W, Bands)
        tokens, H, W = self.tokenizer(x)  # (B, H*W, embed_dim)
        
        # Prepend CLS token
        B = tokens.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        
        for block in self.blocks:
            tokens = block(tokens)
        
        tokens = self.norm(tokens)
        cls_out = tokens[:, 0]  # CLS token output
        return self.head(self.dropout_layer(cls_out))
    
    def extract_features(self, x):
        tokens, H, W = self.tokenizer(x)
        B = tokens.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.norm(tokens)
        return tokens[:, 0]


# Quick test
_test = SSFTT(num_classes=16, num_bands=200, patch_size=15)
_dummy = torch.randn(2, 15, 15, 200)
_out = _test(_dummy)
print(f"SSFTT ready. Test params: {count_params(_test)/1e6:.3f}M, output: {_out.shape}")
del _test, _dummy, _out


SSFTT ready. Test params: 0.173M, output: torch.Size([2, 16])


In [5]:
# =============================================================================
# Cell 5: Training & Evaluation — IDENTICAL protocol to main paper
# =============================================================================
# Same: AdamW, lr=0.001, weight_decay=1e-4, batch_size=64, epochs=30
# Same: 3 seeds {42, 100, 2024}
# Same: metrics (OA, AA, Kappa, Precision, Recall, F1)
# NO SFWOA (transformers use their own cosine schedule)

import gc

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def train_transformer(model, X_train, y_train, X_test, y_test,
                      epochs=100, batch_size=64, lr=0.001, seed=42,
                      model_name="SpectralFormer", dataset_name="IndianPines",
                      early_stop_patience=10):
    """Train a transformer model under identical conditions to main paper."""
    set_seed(seed)
    device = DEVICE
    model = model.to(device)
    
    # Move data to tensors (keep on CPU to prevent OOM)
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_test_t  = torch.tensor(X_test, dtype=torch.float32)
    y_test_t  = torch.tensor(y_test, dtype=torch.long)
    
    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                              batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t),
                              batch_size=batch_size, shuffle=False)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    best_val_acc = 0.0
    best_state = None
    epochs_no_improve = 0
    
    print(f"\n  Training {model_name} on {dataset_name} | seed={seed} | epochs={epochs} | early_stop={early_stop_patience}")
    start_time = time.time()
    
    for epoch in range(epochs):
        # ── Train ──
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            
            running_loss += loss.item() * labels.size(0)
            _, preds = outputs.max(1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()
        
        train_acc = 100.0 * correct / total
        scheduler.step()
        
        # ── Validate ──
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = outputs.max(1)
                val_total += labels.size(0)
                val_correct += (preds == labels).sum().item()
        val_acc = 100.0 * val_correct / val_total
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"    Epoch {epoch+1:3d}/{epochs} | Train {train_acc:.1f}% | Val {val_acc:.1f}%")
        
        # Early stopping
        if early_stop_patience and epochs_no_improve >= early_stop_patience:
            print(f"    Early stopping at epoch {epoch+1} (no improvement for {early_stop_patience} epochs)")
            break
    
    elapsed = time.time() - start_time
    print(f"  Done in {elapsed:.1f}s | Best Val Acc: {best_val_acc:.2f}%")
    
    # Load best weights
    model.load_state_dict(best_state)
    model = model.to(device)
    
    return model, X_test_t, y_test_t, best_val_acc, elapsed


def evaluate_model(model, X_test_t, y_test_t, num_classes, dataset_name, model_name):
    """Full evaluation identical to main paper's metrics."""
    model.eval()
    all_preds = []
    batch_size = 256
    device = next(model.parameters()).device
    for i in range(0, len(X_test_t), batch_size):
        batch = X_test_t[i:i+batch_size].to(device)
        with torch.no_grad():
            outputs = model(batch)
            _, preds = outputs.max(1)
        all_preds.append(preds.cpu().numpy())
    
    y_pred = np.concatenate(all_preds)
    y_true = y_test_t.cpu().numpy()
    
    oa = accuracy_score(y_true, y_pred) * 100
    kappa = cohen_kappa_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0) * 100
    rec = recall_score(y_true, y_pred, average='macro', zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100
    
    cm = confusion_matrix(y_true, y_pred)
    per_class = []
    for i in range(cm.shape[0]):
        if cm[i].sum() > 0:
            per_class.append(cm[i, i] / cm[i].sum() * 100)
        else:
            per_class.append(0.0)
    aa = np.mean(per_class)
    
    metrics = {
        'model': model_name,
        'dataset': dataset_name,
        'OA': round(oa, 2),
        'AA': round(aa, 2),
        'Kappa': round(kappa, 4),
        'Precision': round(prec, 2),
        'Recall': round(rec, 2),
        'F1': round(f1, 2),
        'per_class_acc': [round(p, 2) for p in per_class]
    }
    return metrics


def count_flops_estimate(model, input_shape):
    """Rough FLOPs estimate for comparison."""
    # Use a forward pass with hooks to estimate
    total_flops = 0
    params = count_params(model)
    # Rough estimate: FLOPs ≈ 2 × params × seq_length (for attention-heavy models)
    # More accurate would require torchprofile, but this gives ballpark
    batch = torch.randn(1, *input_shape).to(DEVICE)
    
    # Time-based estimate
    model.eval()
    model.to(DEVICE)
    with torch.no_grad():
        start = time.time()
        for _ in range(100):
            _ = model(batch)
        torch.cuda.synchronize()
        elapsed = (time.time() - start) / 100 * 1000  # ms per inference
    
    del batch
    return params, elapsed


print("Training & evaluation functions ready.")


Training & evaluation functions ready.


In [6]:
# =============================================================================
# Cell 6: RUN ALL EXPERIMENTS
# =============================================================================
# Runs SpectralFormer + SSFTT on all 5 datasets × 3 seeds = 30 runs
# Estimated time: ~30-60 minutes on T4
#
# IMPORTANT: If Kaggle disconnects, just re-run this cell.
# Already-completed results are saved to JSON and will be skipped.

SEEDS = [42, 100, 2024]
WINDOW_SIZE = 15
TRAIN_RATIO = 0.05
EPOCHS = 100
EARLY_STOP_PATIENCE = 20
BATCH_SIZE = 64
LR = 0.001

# ── Which datasets to run ──
# Comment out any you want to skip to save time
DATASETS_TO_RUN = [
    'IndianPines',   # ~200 bands, 16 classes
    'PaviaU',        # ~103 bands, 9 classes
    'Botswana',      # ~145 bands, 14 classes
    'KSC',           # ~176 bands, 13 classes
    # 'WHU_Hi',      # ~274 bands, 16 classes — UNCOMMENT if you uploaded WHU dataset
]

MODELS_TO_RUN = ['SpectralFormer', 'SSFTT']

# ── Storage for all results ──
all_results = {}

for dataset_name in DATASETS_TO_RUN:
    print(f"\n{'='*70}")
    print(f"  DATASET: {dataset_name}")
    print(f"{'='*70}")
    
    # Load and normalize
    X_raw, y_raw = load_dataset(dataset_name)
    X_norm = normalize(X_raw.copy())
    num_bands = X_norm.shape[2]
    num_classes = DATASET_INFO[dataset_name]['num_classes']
    
    for model_name in MODELS_TO_RUN:
        print(f"\n  ── Model: {model_name} ──")
        
        result_file = os.path.join(RESULTS_DIR, f'{model_name}_{dataset_name}_results.json')
        
        # Skip if already completed
        if os.path.exists(result_file):
            print(f"  ⏭ Already completed — loading from {result_file}")
            with open(result_file) as f:
                all_results[f'{model_name}_{dataset_name}'] = json.load(f)
            continue
        
        seed_metrics = []
        seed_times = []
        
        for seed in SEEDS:
            # Create patches (IDENTICAL split to main paper)
            X_train, X_test, y_train, y_test = create_disjoint_patches(
                X_norm, y_raw, window_size=WINDOW_SIZE,
                train_ratio=TRAIN_RATIO, seed=seed
            )
            
            # Create model
            if model_name == 'SpectralFormer':
                model = SpectralFormer(
                    num_classes=num_classes,
                    num_bands=num_bands,
                    patch_size=WINDOW_SIZE,
                    embed_dim=64,
                    depth=4,
                    num_heads=4,
                    dropout=0.1
                )
            elif model_name == 'SSFTT':
                model = SSFTT(
                    num_classes=num_classes,
                    num_bands=num_bands,
                    patch_size=WINDOW_SIZE,
                    embed_dim=64,
                    depth=2,
                    num_heads=4,
                    dropout=0.1
                )
            
            print(f"    Params: {count_params(model)/1e6:.3f}M")
            
            # Train
            model, X_test_t, y_test_t, best_acc, elapsed = train_transformer(
                model, X_train, y_train, X_test, y_test,
                epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
                seed=seed, model_name=model_name, dataset_name=dataset_name,
                early_stop_patience=EARLY_STOP_PATIENCE
            )
            
            # Evaluate
            metrics = evaluate_model(model, X_test_t, y_test_t, num_classes, dataset_name, model_name)
            metrics['seed'] = seed
            metrics['train_time'] = round(elapsed, 2)
            seed_metrics.append(metrics)
            seed_times.append(elapsed)
            
            print(f"    Seed {seed}: OA={metrics['OA']:.2f}%, AA={metrics['AA']:.2f}%, "
                  f"Kappa={metrics['Kappa']:.4f}, F1={metrics['F1']:.2f}%")
            
            # Clean up GPU memory
            del model, X_test_t, y_test_t, X_train, X_test
            torch.cuda.empty_cache()
            gc.collect()
        
        # Compute mean ± std
        summary = {
            'model': model_name,
            'dataset': dataset_name,
            'params_M': round(count_params(
                SpectralFormer(num_classes, num_bands, WINDOW_SIZE) if model_name == 'SpectralFormer'
                else SSFTT(num_classes, num_bands, WINDOW_SIZE)
            ) / 1e6, 3),
            'seeds': SEEDS,
            'per_seed': seed_metrics,
            'mean_OA':   round(np.mean([m['OA'] for m in seed_metrics]), 2),
            'std_OA':    round(np.std([m['OA'] for m in seed_metrics]), 2),
            'mean_AA':   round(np.mean([m['AA'] for m in seed_metrics]), 2),
            'std_AA':    round(np.std([m['AA'] for m in seed_metrics]), 2),
            'mean_Kappa': round(np.mean([m['Kappa'] for m in seed_metrics]), 4),
            'std_Kappa':  round(np.std([m['Kappa'] for m in seed_metrics]), 4),
            'mean_Precision': round(np.mean([m['Precision'] for m in seed_metrics]), 2),
            'mean_Recall':    round(np.mean([m['Recall'] for m in seed_metrics]), 2),
            'mean_F1':        round(np.mean([m['F1'] for m in seed_metrics]), 2),
            'mean_time':      round(np.mean(seed_times), 2),
        }
        
        # Save
        with open(result_file, 'w') as f:
            json.dump(summary, f, indent=2)
        all_results[f'{model_name}_{dataset_name}'] = summary
        
        print(f"\n  ✅ {model_name} on {dataset_name}:")
        print(f"     OA: {summary['mean_OA']:.2f} ± {summary['std_OA']:.2f}%")
        print(f"     AA: {summary['mean_AA']:.2f} ± {summary['std_AA']:.2f}%")
        print(f"     Kappa: {summary['mean_Kappa']:.4f}")
        print(f"     Saved: {result_file}")
        
        torch.cuda.empty_cache()
        gc.collect()

print(f"\n{'='*70}")
print("  ALL EXPERIMENTS COMPLETE")
print(f"{'='*70}")



  DATASET: IndianPines
  Loaded IndianPines: X=(145, 145, 200), y=(145, 145), classes=16

  ── Model: SpectralFormer ──
  Split (seed=42): 505 train, 9744 test
    Params: 0.253M

  Training SpectralFormer on IndianPines | seed=42 | epochs=100 | early_stop=20
    Epoch   1/100 | Train 22.2% | Val 23.9%
    Epoch  10/100 | Train 42.2% | Val 42.0%
    Epoch  20/100 | Train 60.4% | Val 55.9%
    Epoch  30/100 | Train 78.6% | Val 62.6%
    Epoch  40/100 | Train 92.5% | Val 66.9%
    Epoch  50/100 | Train 95.6% | Val 65.2%
    Epoch  60/100 | Train 98.2% | Val 67.5%
    Epoch  70/100 | Train 98.4% | Val 68.8%
    Epoch  80/100 | Train 99.6% | Val 70.3%
    Epoch  90/100 | Train 99.8% | Val 70.3%
    Epoch 100/100 | Train 99.8% | Val 70.3%
  Done in 286.9s | Best Val Acc: 70.40%
    Seed 42: OA=70.40%, AA=63.78%, Kappa=0.6609, F1=65.48%
  Split (seed=100): 505 train, 9744 test
    Params: 0.253M

  Training SpectralFormer on IndianPines | seed=100 | epochs=100 | early_stop=20
    Epoch   1/

In [7]:
# =============================================================================
# Cell 7: Results Summary & Comparison Table
# =============================================================================
# Generates the final comparison table matching the paper's format
# Also measures FLOPs and inference latency

import pandas as pd

print("=" * 80)
print("  TRANSFORMER BASELINE RESULTS SUMMARY")
print("=" * 80)

# ── Load all results ──
result_files = [f for f in os.listdir(RESULTS_DIR) if f.endswith('_results.json')]
all_loaded = {}
for rf in sorted(result_files):
    with open(os.path.join(RESULTS_DIR, rf)) as f:
        data = json.load(f)
        key = f"{data['model']}_{data['dataset']}"
        all_loaded[key] = data

# ── Print per-dataset comparison ──
for dataset_name in DATASETS_TO_RUN:
    print(f"\n{'─'*60}")
    print(f"  {dataset_name}")
    print(f"{'─'*60}")
    
    rows = []
    for model_name in MODELS_TO_RUN:
        key = f"{model_name}_{dataset_name}"
        if key in all_loaded:
            d = all_loaded[key]
            rows.append({
                'Model': model_name,
                'OA (%)': f"{d['mean_OA']:.2f} ± {d['std_OA']:.2f}",
                'AA (%)': f"{d['mean_AA']:.2f} ± {d['std_AA']:.2f}",
                'Kappa': f"{d['mean_Kappa']:.4f}",
                'F1 (%)': f"{d['mean_F1']:.2f}",
                'Params (M)': f"{d['params_M']:.3f}",
                'Time (s)': f"{d['mean_time']:.1f}"
            })
    
    if rows:
        df = pd.DataFrame(rows)
        print(df.to_string(index=False))

# ── Measure inference latency ──
print(f"\n{'─'*60}")
print("  INFERENCE LATENCY (ms per patch)")
print(f"{'─'*60}")

for dataset_name in DATASETS_TO_RUN:
    X_raw, y_raw = load_dataset(dataset_name)
    num_bands = X_raw.shape[2]
    num_classes = DATASET_INFO[dataset_name]['num_classes']
    
    for model_name in MODELS_TO_RUN:
        if model_name == 'SpectralFormer':
            model = SpectralFormer(num_classes, num_bands, WINDOW_SIZE).to(DEVICE)
        else:
            model = SSFTT(num_classes, num_bands, WINDOW_SIZE).to(DEVICE)
        
        model.eval()
        dummy = torch.randn(1, WINDOW_SIZE, WINDOW_SIZE, num_bands).to(DEVICE)
        
        # Warmup
        with torch.no_grad():
            for _ in range(20):
                _ = model(dummy)
        torch.cuda.synchronize()
        
        # Measure
        start = time.time()
        with torch.no_grad():
            for _ in range(200):
                _ = model(dummy)
        torch.cuda.synchronize()
        latency_ms = (time.time() - start) / 200 * 1000
        
        print(f"  {model_name:20s} on {dataset_name:15s}: {latency_ms:.2f} ms/patch, "
              f"Params: {count_params(model)/1e6:.3f}M")
        
        del model, dummy
        torch.cuda.empty_cache()

# ── Save combined results for paper update ──
combined = {
    'experiment_config': {
        'seeds': SEEDS,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'window_size': WINDOW_SIZE,
        'train_ratio': TRAIN_RATIO,
        'optimizer': 'AdamW',
        'scheduler': 'CosineAnnealingLR',
        'note': 'Identical protocol to main paper (no SFWOA for transformers, uses cosine schedule)'
    },
    'results': all_loaded
}

combined_path = os.path.join(RESULTS_DIR, 'transformer_baselines_combined.json')
with open(combined_path, 'w') as f:
    json.dump(combined, f, indent=2)

print(f"\n✅ Combined results saved: {combined_path}")
print("\n📋 COPY THESE NUMBERS INTO create_paper.js Table 11!")
print("   Then regenerate the docx with: node create_paper.js")


  TRANSFORMER BASELINE RESULTS SUMMARY

────────────────────────────────────────────────────────────
  IndianPines
────────────────────────────────────────────────────────────
         Model       OA (%)       AA (%)  Kappa F1 (%) Params (M) Time (s)
SpectralFormer 69.60 ± 1.16 60.31 ± 2.49 0.6515  62.25      0.253    278.8
         SSFTT 95.69 ± 0.69 86.85 ± 5.05 0.9509  87.76      0.121    476.9

────────────────────────────────────────────────────────────
  PaviaU
────────────────────────────────────────────────────────────
         Model       OA (%)       AA (%)  Kappa F1 (%) Params (M) Time (s)
SpectralFormer 95.50 ± 0.58 93.38 ± 0.69 0.9403  93.46      0.247    468.7
         SSFTT 99.52 ± 0.05 99.04 ± 0.19 0.9937  99.06      0.121   1339.3

────────────────────────────────────────────────────────────
  Botswana
────────────────────────────────────────────────────────────
         Model       OA (%)       AA (%)  Kappa F1 (%) Params (M) Time (s)
SpectralFormer 82.13 ± 0.70 77.56